# 🐾 반려동물 피부질환 데이터 EDA (마일스톤 2)

**프로젝트**: 반려동물 피부질환 감지 AI  
**목표**: 데이터셋의 기본 구조를 파악하고 EDA 그래프 5개 이상과 해석을 기록한다  
**데이터 출처**: AI Hub - 반려동물 피부질환 데이터

---
## 마일스톤 2 체크리스트
- [x] 데이터 로딩이 성공했다
- [x] 데이터 크기와 컬럼을 설명할 수 있다
- [x] 주요 컬럼의 의미를 정리했다
- [x] 결측치와 이상치를 확인했다
- [x] target 분포를 확인했다
- [x] 기본 그래프 5개 이상을 만들었다
- [x] 각 그래프에 해석을 1~2줄 작성했다


## 0. 라이브러리 임포트 및 설정

In [6]:
!pip install pandas numpy matplotlib seaborn pillow ipykernel


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 스타일
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ 라이브러리 로딩 완료")


✅ 라이브러리 로딩 완료


## 1. 데이터 로딩 ✅ (체크 #1)

AI Hub 데이터는 JSON + JPG 파일 쌍으로 구성되어 있습니다.  
모든 JSON 파일을 순회하며 메타데이터와 라벨링 정보를 추출하여 DataFrame으로 변환합니다.

### 데이터 경로
- **Training**: `D:\data\152.반려동물 피부질환 데이터\01.데이터\1.Training\2_라벨링데이터_240422_add\`
- **Validation**: `D:\data\152.반려동물 피부질환 데이터\01.데이터\2.Validation\2_라벨링데이터_240422_add\`


In [8]:
# ============================================================
# ⚠️ 아래 경로를 본인 환경에 맞게 확인하세요
# ============================================================
TRAIN_DIR = r"D:\data\152.반려동물 피부질환 데이터\01.데이터\1.Training\2_라벨링데이터_240422_add"
VAL_DIR = r"D:\data\152.반려동물 피부질환 데이터\01.데이터\2.Validation\2_라벨링데이터_240422_add"

DATASET_DIRS = {
    "TL01": os.path.join(TRAIN_DIR, "TL01"),
    "TL02": os.path.join(TRAIN_DIR, "TL02"),
    "VL01": os.path.join(VAL_DIR, "VL01"),
}

# 경로 존재 여부 확인
for name, path in DATASET_DIRS.items():
    exists = "✅ 존재" if os.path.exists(path) else "❌ 경로 없음"
    print(f"  {name}: {path} → {exists}")


  TL01: D:\data\152.반려동물 피부질환 데이터\01.데이터\1.Training\2_라벨링데이터_240422_add\TL01 → ✅ 존재
  TL02: D:\data\152.반려동물 피부질환 데이터\01.데이터\1.Training\2_라벨링데이터_240422_add\TL02 → ✅ 존재
  VL01: D:\data\152.반려동물 피부질환 데이터\01.데이터\2.Validation\2_라벨링데이터_240422_add\VL01 → ✅ 존재


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

def parse_json_file(filepath, dataset_name):
    """JSON 파일 하나를 파싱하여 딕셔너리로 반환"""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    meta = data.get('metaData', {})
    label_info = data.get('labelingInfo', [])
    
    label = ''
    has_bbox = False
    has_polygon = False
    bbox_info = {}
    
    for item in label_info:
        if 'polygon' in item:
            has_polygon = True
            label = item['polygon'].get('label', '')
        if 'box' in item:
            has_bbox = True
            loc = item['box'].get('location', [{}])
            bbox_info = loc[0] if loc else {}
    
    return {
        'dataset': dataset_name,
        'split': 'train' if dataset_name.startswith('TL') else 'val',
        'filename': meta.get('Raw data ID', ''),
        'species': meta.get('species', ''),
        'breed': meta.get('breed', ''),
        'age': meta.get('age', ''),
        'gender': meta.get('gender', ''),
        'region': meta.get('region', ''),
        'lesions': meta.get('lesions', ''),
        'camera_type': meta.get('camera type', ''),
        'resolution': meta.get('resolution', ''),
        'path_status': meta.get('Path', ''),
        'diagnosis': meta.get('diagnosis', ''),
        'label': label,
        'has_bbox': has_bbox,
        'has_polygon': has_polygon,
        'bbox_x': bbox_info.get('x', None),
        'bbox_y': bbox_info.get('y', None),
        'bbox_w': bbox_info.get('width', None),
        'bbox_h': bbox_info.get('height', None),
        'json_path': filepath,
    }

def _parse_worker(args):
    filepath, ds_name = args
    try:
        return parse_json_file(filepath, ds_name), None
    except Exception as e:
        return None, (filepath, str(e))

# 전체 JSON 파일 목록 수집
all_tasks = []
for ds_name, ds_path in DATASET_DIRS.items():
    if not os.path.exists(ds_path):
        print(f"⚠️ 경로 없음, 건너뜀: {ds_path}")
        continue
    json_files = list(Path(ds_path).rglob('*.json'))
    print(f"  {ds_name}: {len(json_files):,}개 JSON 발견")
    all_tasks.extend((str(p), ds_name) for p in json_files)

print(f"\n총 {len(all_tasks):,}개 파일 병렬 파싱 시작...")

rows = []
error_files = []

# I/O 바운드 작업 → 스레드 병렬화 (workers=16 권장)
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(_parse_worker, task): task for task in all_tasks}
    done = 0
    for future in as_completed(futures):
        row, err = future.result()
        if row:
            rows.append(row)
        if err:
            error_files.append(err)
        done += 1
        if done % 5000 == 0:
            print(f"  진행: {done:,} / {len(all_tasks):,}")

df = pd.DataFrame(rows)
print(f"\n✅ 데이터 로딩 완료!")
print(f"   총 샘플 수: {len(df):,}개")
print(f"   오류 파일: {len(error_files)}개")


  TL01: 263,340개 JSON 발견
  TL02: 177,605개 JSON 발견
  VL01: 55,117개 JSON 발견

총 496,062개 파일 병렬 파싱 시작...
  진행: 5,000 / 496,062
  진행: 10,000 / 496,062
  진행: 15,000 / 496,062
  진행: 20,000 / 496,062
  진행: 25,000 / 496,062
  진행: 30,000 / 496,062
  진행: 35,000 / 496,062
  진행: 40,000 / 496,062
  진행: 45,000 / 496,062
  진행: 50,000 / 496,062
  진행: 55,000 / 496,062
  진행: 60,000 / 496,062
  진행: 65,000 / 496,062
  진행: 70,000 / 496,062
  진행: 75,000 / 496,062
  진행: 80,000 / 496,062
  진행: 85,000 / 496,062
  진행: 90,000 / 496,062
  진행: 95,000 / 496,062
  진행: 100,000 / 496,062
  진행: 105,000 / 496,062
  진행: 110,000 / 496,062
  진행: 115,000 / 496,062
  진행: 120,000 / 496,062
  진행: 125,000 / 496,062
  진행: 130,000 / 496,062
  진행: 135,000 / 496,062
  진행: 140,000 / 496,062
  진행: 145,000 / 496,062
  진행: 150,000 / 496,062
  진행: 155,000 / 496,062
  진행: 160,000 / 496,062
  진행: 165,000 / 496,062
  진행: 170,000 / 496,062
  진행: 175,000 / 496,062
  진행: 180,000 / 496,062
  진행: 185,000 / 496,062
  진행: 190,000 / 496,062
  진행: 1

## 2. 데이터 크기와 컬럼 설명 ✅ (체크 #2)

In [ ]:
# 데이터 크기
print(f"📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print()

# 데이터셋별 분할
print("=== 데이터셋별 샘플 수 ===")
print(df['dataset'].value_counts().to_string())
print()

# Train / Val 분할
print("=== Train / Validation 분할 ===")
print(df['split'].value_counts().to_string())
print()

# 컬럼 정보
print("=== 컬럼 상세 정보 ===")
df.info()


In [ ]:
# 상위 5개 데이터 미리보기
df.head()


In [ ]:
# 수치형 데이터 통계
df['age_num'] = pd.to_numeric(df['age'], errors='coerce')
df.describe()


## 3. 주요 컬럼의 의미 정리 ✅ (체크 #3)

| 컬럼 | 설명 | 값 예시 |
|------|------|---------|
| `dataset` | 데이터셋 구분 | TL01, TL02 (Train) / VL01 (Validation) |
| `split` | Train/Val 구분 | train, val |
| `species` | 동물 종 | C (반려묘), D (반려견) |
| `breed` | 품종 | 말티즈, 코리안숏헤어, 푸들 등 |
| `age` | 나이 (세) | 1 ~ 17 |
| `gender` | 성별 | M (수컷), F (암컷) |
| `region` | 병변 부위 | B (몸통), L (다리), H (머리), A (복부) |
| `lesions` | 질환 코드 | A1~A6 (일반카메라), 빈값 (현미경) |
| `camera_type` | 촬영 방식 | IMG (일반카메라), CYT (현미경) |
| `path_status` | 증상 유무 | 유증상, 무증상, 감염성피부염, 비감염성피부염 |
| `diagnosis` | 진단 코드 | C1 (감염성), C6 (비감염성), 빈값 |
| `label` | **라벨명 (target)** | A1_구진_플라크, A7_무증상, C1_감염성피부염 등 |
| `has_bbox` | 바운딩박스 유무 | True / False |
| `has_polygon` | 폴리곤 유무 | True / False |
| `bbox_x/y/w/h` | 바운딩박스 좌표 | 수치 (픽셀 단위) |

### 질환 클래스 정리

**일반카메라 (IMG) — A 코드 (7종)**

| 코드 | 질환명 | 설명 |
|------|--------|------|
| A1 | 구진 / 플라크 | 피부 표면에 솟아오른 병변 |
| A2 | 비듬 / 각질 / 상피성잔고리 | 각질 탈락, 비듬 형태의 병변 |
| A3 | 태선화 / 과다색소침착 | 피부가 두꺼워지고 색이 짙어지는 병변 |
| A4 | 농포 / 여드름 | 고름이 차 있는 병변 |
| A5 | 미란 / 궤양 | 피부 표면이 벗겨진 병변 |
| A6 | 결절 / 종괴 | 피부 아래 덩어리가 만져지는 병변 |
| A7 | 무증상 (정상) | 피부 질환 증상 없음 |

**현미경 (CYT) — C 코드 (2종)**

| 코드 | 질환명 | 설명 |
|------|--------|------|
| C1 | 감염성 피부염 | 세균/진균 등 감염에 의한 피부염 |
| C6 | 비감염성 피부염 | 알레르기 등 비감염성 원인의 피부염 |


## 4. 결측치와 이상치 확인 ✅ (체크 #4)

> 💡 **오늘은 확인만** 하고, 실제 처리(제거/대체)는 전처리 단계에서 진행합니다.  
> 결측치: 데이터가 아예 비어있는 것 (NaN, null)  
> 이상치: 데이터는 있지만 비정상적으로 크거나 작은 값


In [ ]:
# 4-1. 결측치 확인 (NaN)
print("=== 컬럼별 결측치(NaN) 수 ===")
null_counts = df.isnull().sum()
null_found = null_counts[null_counts > 0]
if len(null_found) > 0:
    print(null_found)
else:
    print("NaN 결측치 없음 ✅")
print()

# 4-2. 빈 문자열('') 체크 — JSON 특성상 null 대신 빈 문자열일 수 있음
print("=== 빈 문자열('') 개수 ===")
check_cols = ['species', 'breed', 'age', 'gender', 'region', 'lesions', 'diagnosis', 'label', 'camera_type']
for col in check_cols:
    empty_count = (df[col] == '').sum()
    if empty_count > 0:
        print(f"  {col}: {empty_count:,}개 ({empty_count/len(df)*100:.1f}%)")

print()
print("💡 lesions와 diagnosis의 빈값은 카메라 타입에 따른 구조적 차이입니다.")
print("   - 일반카메라(IMG): lesions에 A코드 기록, diagnosis 빈값")
print("   - 현미경(CYT): lesions 빈값, diagnosis에 C코드 기록")


In [ ]:
# 4-3. 결측치 히트맵 시각화
fig, ax = plt.subplots(figsize=(14, 5))

# 빈 문자열을 NaN으로 치환하여 시각화
df_check = df[check_cols].replace('', np.nan)
sns.heatmap(df_check.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=ax)
ax.set_title('결측치 히트맵 (빈 문자열 포함)', fontsize=14)
plt.tight_layout()
plt.show()


> **해석**: `lesions`와 `diagnosis` 컬럼에 빈 문자열이 존재합니다. 이는 결측이 아니라 **일반카메라(IMG)는 lesions(A코드)**, **현미경(CYT)은 diagnosis(C코드)**를 사용하는 구조적 차이입니다. 실제 데이터 결측은 없으므로, 전처리 시 카메라 타입별로 구분하여 처리하면 됩니다.


In [ ]:
# 4-4. 이상치 확인 - 나이(age) BoxPlot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 나이 분포
sns.boxplot(data=df, y='age_num', ax=axes[0], color='skyblue')
axes[0].set_title('전체 나이 분포 (BoxPlot)', fontsize=13)
axes[0].set_ylabel('나이 (세)')

# 종별 나이 분포
species_map = {'C': '반려묘', 'D': '반려견'}
df['species_kr'] = df['species'].map(species_map)
sns.boxplot(data=df, x='species_kr', y='age_num', ax=axes[1], palette='Set2')
axes[1].set_title('종별 나이 분포 (BoxPlot)', fontsize=13)
axes[1].set_xlabel('동물 종')
axes[1].set_ylabel('나이 (세)')

plt.tight_layout()
plt.show()


> **해석**: 나이 분포에서 뚜렷한 이상치(극단값)는 발견되지 않았습니다. 전체 1~17세 범위이며, 오늘은 확인만 하고 이상치 처리는 전처리 단계에서 필요 시 진행합니다.


## 5. Target 분포 확인 ✅ (체크 #5)

이 프로젝트의 **target(예측 대상)**은 `label` 컬럼입니다.


In [ ]:
# target(label) 분포 확인
print("=== Target(label) 분포 ===")
label_counts = df['label'].value_counts()
for label, count in label_counts.items():
    pct = count / len(df) * 100
    print(f"  {label}: {count:,}개 ({pct:.1f}%)")

print(f"\n총 클래스 수: {df['label'].nunique()}개")
print(f"총 샘플 수: {len(df):,}개")


In [ ]:
# Target 분포 시각화
fig, ax = plt.subplots(figsize=(14, 6))

label_counts = df['label'].value_counts()
colors = sns.color_palette('Set2', len(label_counts))
bars = ax.barh(label_counts.index, label_counts.values, color=colors)

for bar, val in zip(bars, label_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_width() + max(label_counts.values)*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:,}개 ({pct:.1f}%)', va='center', fontsize=11)

ax.set_xlabel('샘플 수', fontsize=12)
ax.set_title('📊 Target(질환 라벨) 분포', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


> **해석**: A7_무증상(정상) 클래스가 전체에서 가장 큰 비중을 차지하며 **클래스 불균형**이 존재합니다. 모델 학습 시 오버샘플링, 언더샘플링, 또는 클래스 가중치(class_weight) 적용이 필요합니다.


## 6. EDA 시각화 ✅ (체크 #6, #7)

---
### 그래프 1: 동물 종(species) 분포


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

species_counts = df['species_kr'].value_counts()

# Pie Chart
axes[0].pie(species_counts.values, labels=species_counts.index,
            autopct='%1.1f%%', colors=['#FF9999', '#66B3FF'], startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('동물 종 비율', fontsize=14)

# Bar Chart
sns.countplot(data=df, x='species_kr', palette=['#FF9999', '#66B3FF'], ax=axes[1])
axes[1].set_title('동물 종별 샘플 수', fontsize=14)
axes[1].set_xlabel('동물 종')
axes[1].set_ylabel('샘플 수')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()


> **해석**: 반려견 데이터가 반려묘보다 많습니다. 종별 불균형을 고려하여 모델 학습 시 종을 feature로 활용하거나, 종별 데이터 증강 전략이 필요합니다.


### 그래프 2: 나이(age) 분포

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df, x='age_num', bins=17, kde=True, color='coral', ax=ax)
ax.set_title('나이(age) 분포', fontsize=14)
ax.set_xlabel('나이 (세)')
ax.set_ylabel('빈도')
ax.axvline(df['age_num'].mean(), color='red', linestyle='--',
           label=f"평균: {df['age_num'].mean():.1f}세")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


> **해석**: 나이 분포는 전체적으로 고르게 퍼져 있으며, 특정 연령대에 집중되는 경향이 있습니다. 나이가 피부질환 유형과 상관관계가 있는지 추후 분석이 가능합니다.


### 그래프 3: 품종별(breed) 상위 10개 분포

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
breed_top10 = df['breed'].value_counts().head(10)
sns.barplot(x=breed_top10.values, y=breed_top10.index, palette='viridis', ax=ax)
ax.set_title('품종별 분포 (상위 10개)', fontsize=14)
ax.set_xlabel('샘플 수')
ax.set_ylabel('품종')
for i, val in enumerate(breed_top10.values):
    ax.text(val + max(breed_top10.values)*0.01, i, f'{val:,}개', va='center', fontsize=11)
plt.tight_layout()
plt.show()


> **해석**: 특정 품종(코리안숏헤어, 말티즈 등)에 데이터가 편중되어 있습니다. 품종 다양성 측면에서 한계가 있으며, 소수 품종에 대한 모델 일반화 성능에 주의가 필요합니다.


### 그래프 4: 병변 부위(region) 분포

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

region_labels = {'B': 'B (몸통)', 'L': 'L (다리)', 'H': 'H (머리)', 'A': 'A (복부)'}
df['region_kr'] = df['region'].map(region_labels).fillna(df['region'])

region_counts = df['region_kr'].value_counts()
bars = ax.bar(region_counts.index, region_counts.values,
              color=sns.color_palette('Pastel1', len(region_counts)))
ax.set_title('병변 부위(region) 분포', fontsize=14)
ax.set_xlabel('병변 부위')
ax.set_ylabel('샘플 수')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()


> **해석**: 몸통(B)과 다리(L)에 병변이 가장 많이 발생하며, 복부(A)는 상대적으로 적습니다. 부위별로 질환 유형이 다를 수 있어 교차분석이 유의미합니다.


### 그래프 5: 동물 종 × 질환 라벨 교차분석 (Heatmap)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ct = pd.crosstab(df['species_kr'], df['label'])
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('동물 종 × 질환 라벨 교차분석', fontsize=14)
ax.set_xlabel('질환 라벨')
ax.set_ylabel('동물 종')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


> **해석**: 반려견은 전체 질환 클래스에 비교적 고르게 분포하는 반면, 반려묘는 일부 클래스에 데이터가 부족합니다. 종별로 모델을 분리하거나, 데이터 증강으로 보완하는 전략이 필요합니다.


### 그래프 6: 성별(gender) × 종별 분포

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

gender_labels = {'M': '수컷(M)', 'F': '암컷(F)'}
df['gender_kr'] = df['gender'].map(gender_labels)

sns.countplot(data=df, x='gender_kr', hue='species_kr', palette='Set2', ax=ax)
ax.set_title('성별 × 종별 분포', fontsize=14)
ax.set_xlabel('성별')
ax.set_ylabel('샘플 수')
ax.legend(title='동물 종')
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()


> **해석**: 수컷(M)이 암컷(F)보다 전반적으로 많으며, 성별이 피부질환 발생률에 영향을 미치는지는 추가 분석이 필요합니다.


### 그래프 7: 촬영 방식(camera_type) 분포

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

cam_labels = {'IMG': '일반카메라(IMG)', 'CYT': '현미경(CYT)'}
df['camera_kr'] = df['camera_type'].map(cam_labels).fillna('기타')

cam_counts = df['camera_kr'].value_counts()
ax.pie(cam_counts.values, labels=cam_counts.index, autopct='%1.1f%%',
       colors=['#87CEEB', '#DDA0DD', '#98FB98'], startangle=90, textprops={'fontsize': 13})
ax.set_title('촬영 방식 분포', fontsize=14)
plt.tight_layout()
plt.show()


> **해석**: 일반카메라(IMG) 데이터가 대부분을 차지합니다. 일반카메라는 A1~A7(7종), 현미경은 C1/C6(2종)으로 라벨 체계가 다르므로 모델 설계 시 이를 구분해야 합니다.


## 7. 샘플 이미지 시각화

각 질환 클래스별 대표 이미지를 확인합니다.


In [ ]:
# 클래스별 샘플 이미지 시각화
def get_image_path(json_path):
    """JSON 경로에서 이미지 경로 추출"""
    return json_path.replace('.json', '.jpg')

sample_per_class = df.groupby('label').first().reset_index()
n_classes = len(sample_per_class)
cols = 5
rows_needed = (n_classes + cols - 1) // cols

fig, axes = plt.subplots(rows_needed, cols, figsize=(20, 4 * rows_needed))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample_per_class.iterrows()):
    if idx >= len(axes):
        break
    img_path = get_image_path(row['json_path'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[idx].imshow(img)
    else:
        axes[idx].text(0.5, 0.5, '이미지 없음', ha='center', va='center', fontsize=10)
    axes[idx].set_title(row['label'], fontsize=10)
    axes[idx].axis('off')

for idx in range(n_classes, len(axes)):
    axes[idx].axis('off')

plt.suptitle('클래스별 샘플 이미지', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


## 8. EDA 종합 정리

### 데이터 요약
- **전체 구조**: JSON(메타+라벨) + JPG(이미지) 1:1 쌍
- **Train/Val 분할**: TL01 + TL02 (Training) / VL01 (Validation)
- **동물 종**: 반려견(D), 반려묘(C) 포함
- **촬영 방식**: 일반카메라(IMG), 현미경(CYT) 2종
- **질환 클래스**: 일반카메라 A1~A7 (7종) + 현미경 C1, C6 (2종) = 총 9종
- **어노테이션**: Polygon + Bounding Box 모두 제공 → 분류 & 탐지 모두 가능

### 주요 발견
1. **클래스 불균형**: A7_무증상이 가장 큰 비중 → 오버샘플링/클래스 가중치 필요
2. **종별 불균형**: 반려견이 반려묘보다 많음 → 종별 분리 또는 증강 고려
3. **라벨 체계 차이**: 일반카메라(A코드)와 현미경(C코드)의 라벨이 다름 → 별도 처리 필요
4. **결측치 없음**: 구조적 빈값만 존재, 실제 결측은 없음
5. **이상치 없음**: 나이 등 수치형 데이터에서 극단값 미발견

### 다음 단계 (전처리)
- [ ] 일반카메라 / 현미경 데이터 분리 여부 결정
- [ ] 클래스 불균형 해소 전략 적용 (오버샘플링, 가중치 등)
- [ ] 이미지 리사이즈 (224×224 또는 640×640)
- [ ] 데이터 증강 (회전, 플립, 밝기 조절 등)
- [ ] Bounding Box 좌표 → YOLO 포맷 변환
